## Insider Threat Radar – Behavior Analytics for Employee Risk Prediction

### Prediction 

#### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib

#### Load Model

In [2]:
model = joblib.load("../Model/random_forest.pkl")

print("Final model loaded ✅")

Final model loaded ✅


#### Load Feature File

In [3]:
data = pd.read_csv("../Dataset/final_features.csv")
data.head()

,user,total_emails,total_attachments,avg_email_size,off_hour_emails,weekend_emails,employee_name,user_id,O,C,E,A,N
0,AAE0190,4711,1780,30020.394184,499,0,August Armando Evans,AAE0190,36,30,14,50,29
1,AAF0535,480,364,30397.402083,1,0,Athena Amelia Foreman,AAF0535,17,21,36,33,31
2,AAF0791,3012,0,29958.497676,377,0,Aladdin Abraham Foley,AAF0791,14,40,40,50,34
3,AAL0706,336,145,29828.181548,54,0,April Alika Levy,AAL0706,37,14,28,13,25
4,AAM0658,659,613,29895.532625,146,0,Abel Adam Morton,AAM0658,43,35,37,36,22


#### Select Features

In [4]:
feature_cols = [
    'total_emails','total_attachments','avg_email_size',
    'off_hour_emails','weekend_emails',
    'O','C','E','A','N'
]

X = data[feature_cols]

#### Predict (Normal vs Anomaly)

In [5]:
pred = model.predict(X)

data['prediction'] = pred
data['prediction_label'] = data['prediction'].map(
    {0:"Normal", 1:"Suspicious"}
)

data[['user','prediction_label']].head()

,user,prediction_label
0,AAE0190,Normal
1,AAF0535,Normal
2,AAF0791,Normal
3,AAL0706,Normal
4,AAM0658,Normal


#### Risk Probability

In [6]:
probs = model.predict_proba(X)[:,1]

data['risk_probability'] = probs

#### Risk Score (0–100)

In [7]:
data['risk_score'] = (data['risk_probability']*100).round(2)

#### Risk Category

In [8]:
def risk_level(score):
    if score < 40:
        return "Low"
    elif score < 70:
        return "Medium"
    else:
        return "High"

data['risk_level'] = data['risk_score'].apply(risk_level)

data[['user','risk_score','risk_level']].head()

,user,risk_score,risk_level
0,AAE0190,0.0,Low
1,AAF0535,0.0,Low
2,AAF0791,0.0,Low
3,AAL0706,0.0,Low
4,AAM0658,0.0,Low


#### Show High Risk Users

In [9]:
high_risk = data[data['risk_level']=="High"]

high_risk[['user','risk_score','risk_level']].head(10)

,user,risk_score,risk_level
83,ATE0869,92.5,High
141,BTW0005,76.5,High
265,DLM0051,90.0,High
380,HAD0246,72.5,High
418,HPH0075,83.0,High
425,HSB0196,74.0,High
430,HTH0007,77.5,High
488,JDC0030,76.0,High
541,KBP0008,81.0,High
590,LBF0214,82.0,High


#### Save Prediction Output

In [11]:
data.to_csv("../Dataset/prediction_output.csv", index=False)
high_risk.to_csv("../Dataset/high_risk_prediction.csv", index=False)

print("Prediction results saved ✅")

Prediction results saved ✅


### 📌 SINGLE USER PREDICTION

**Prediction Function**

In [14]:
# Choose any user from dataset
single_user_id = "HPH0075"

single_user = data[data['user'] == single_user_id]

print("Selected User Data:")
display(single_user)

# Feature columns
X_single = single_user[feature_cols]

# Prediction
pred = model.predict(X_single)[0]
prob = model.predict_proba(X_single)[0][1]

label = "Suspicious" if pred==1 else "Normal"

print("\n🔍 SINGLE USER RESULT")
print("User:", single_user_id)
print("Prediction:", label)
print("Risk Score:", round(prob*100,2))

Selected User Data:


,user,total_emails,total_attachments,avg_email_size,off_hour_emails,weekend_emails,employee_name,user_id,O,C,E,A,N,prediction,prediction_label,risk_probability,risk_score,risk_level
418,HPH0075,7435,4528,29988.672091,1179,1075,Harper Price Harris,HPH0075,40,42,48,25,28,1,Suspicious,0.83,83.0,High



🔍 SINGLE USER RESULT
User: HPH0075
Prediction: Suspicious
Risk Score: 83.0
